# Dyson Protocol Bank Module Guide

This notebook provides quick, copy-paste-ready examples for the Bank module:
- Query balances via CLI and via Dyslang (read-only)
- Send coins via CLI and via Dyslang (transaction)

It complements the high-level context in `agents.md` and `README.md`.



## 1. Prerequisites

- You have `dysond` installed and connected to a network (see `README.md`).
- You have at least one local key (e.g., `alice`) with funds.
- Replace account names and addresses as needed.


## 2. Query Balance Examples

### 2.1 Query via CLI
Use the standard bank balance query. Replace `alice` with your key name if different.



In [ ]:
import json

# Get alice address
[ALICE] = get_ipython().getoutput("dysond keys show -a alice")
# Query balance
! dysond query bank balance {ALICE} udys -o json | jq -M


### 2.2 Query via Dyslang (read-only)
Runs a minimal script inline using `--extra-code` to query the balance from within the sandbox.



In [ ]:
import tempfile, shlex

# Get alice address
[ALICE] = get_ipython().getoutput("dysond keys show -a alice")

extra_code = """
from dys import _query

def query_balance(address, denom="udys"):
    return _query({
        "@type": "/cosmos.bank.v1beta1.QueryBalanceRequest",
        "address": address,
        "denom": denom,
    })
"""
with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False) as f:
    f.write(extra_code)
    extra_path = f.name

args = shlex.quote(json.dumps([ALICE, "udys"]))
! dysond query script run --script-address {ALICE} --executor-address {ALICE} --function-name query_balance --args {args} --extra-code-path {extra_path} -o json | jq -M


## 3. Send Coins Examples

### 3.1 Send via CLI
Send 123 udys from `alice` to `bob`.



In [ ]:
import json

[ALICE] = get_ipython().getoutput("dysond keys show -a alice")
[BOB] = get_ipython().getoutput("dysond keys show -a bob")

! dysond tx bank send alice {BOB} 123udys -y -o json | dysond query wait-tx -o json | jq -M


### 3.2 Send via Dyslang (transaction)
Use an inline script executed via `dysond tx script exec` to send funds.



In [ ]:
import tempfile, shlex

[ALICE] = get_ipython().getoutput("dysond keys show -a alice")
[BOB] = get_ipython().getoutput("dysond keys show -a bob")

extra_code = """
from dys import _msg, get_executor_address

def send_to(recipient, amount, denom="udys"):
    return _msg({
        "@type": "/cosmos.bank.v1beta1.MsgSend",
        "from_address": get_executor_address(),
        "to_address": recipient,
        "amount": [{"denom": denom, "amount": str(amount)}],
    })
"""
with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False) as f:
    f.write(extra_code)
    extra_path = f.name

args = shlex.quote(json.dumps([BOB, 123, "udys"]))
! dysond tx script exec --from alice --script-address {ALICE} --function-name send_to --args {args} --extra-code-path {extra_path} -y -o json | dysond query wait-tx -o json | jq -M
